# LeetCode 1101 — The Earliest Moment When Everyone Become Friends bằng Python

Link đề: https://leetcode.com/problems/the-earliest-moment-when-everyone-become-friends/ · Độ khó: **Medium** · Chủ đề: Array, Union Find, Sorting, Graph

> **Premium:** đây là bài **Premium** (🔒). Đề trong notebook được viết lại theo hiểu biết và theo bản mô tả trong kho tham khảo; nếu bạn có tài khoản Premium hãy **đối chiếu lại** đề gốc (đặc biệt: "quen biết" là quan hệ **bắc cầu**; trả về **thời điểm** của log làm cả nhóm liên thông, hoặc `-1` nếu không bao giờ).

Skeleton LeetCode:

```python
class Solution:
    def earliestAcq(self, logs: List[List[int]], n: int) -> int:
        ...
```

Ghi chú: đáp án là **một số nguyên duy nhất**. Các log `[timestamp, x, y]` có thể **không được sắp xếp theo thời gian**; timestamp đôi một khác nhau. Điểm cốt lõi: sắp xếp log theo thời gian, dùng **Union-Find** gộp hai nhóm bạn bè; thời điểm đầu tiên số nhóm còn `1` là đáp án. Hoặc: đáp án chính là **trọng số lớn nhất trong cây khung nhỏ nhất** (bottleneck).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: `n` người, các sự kiện kết bạn theo thời gian, "quen biết" là bắc cầu; tìm thời điểm sớm nhất mọi người quen biết nhau (đồ thị liên thông).
2. Nắm công cụ: sắp xếp theo thời gian, Union-Find (tìm gốc + nén đường đi + hợp theo kích thước), đếm số thành phần, BFS/DFS, tìm kiếm nhị phân trên chỉ số log, thuật toán Prim, Floyd–Warshall dạng minimax.
3. Đi từ **vét cạn theo từng tiền tố log** → **Union-Find sau khi sắp xếp** → **nhị phân + BFS** → **Prim (bottleneck của cây khung)** → **Floyd–Warshall minimax**.
4. Hiểu mấu chốt: đồ thị chỉ **thêm cạnh** theo thời gian nên tính liên thông là **đơn điệu**; thời điểm đáp án là cạnh làm số thành phần giảm xuống `1`. Đáp án cũng bằng `max` trọng số cạnh của **cây khung nhỏ nhất** theo thời gian.
5. Tránh lỗi kinh điển: quên sắp xếp log, giảm số thành phần cả khi hai người đã chung nhóm, quên trả `-1`, gộp mà không tìm gốc, trả về chỉ số thay vì timestamp.

## 2. Đề bài & ví dụ

Có `n` người trong một nhóm xã hội, gắn nhãn `0..n-1`. Cho mảng `logs`, `logs[i] = [timestamp_i, x_i, y_i]` nghĩa là `x_i` và `y_i` **trở thành bạn** tại thời điểm `timestamp_i`.

Tình bạn có tính **đối xứng**. Ngoài ra `a` **quen biết** `b` nếu `a` là bạn của `b`, hoặc `a` là bạn của một người quen biết `b` (quan hệ **bắc cầu**).

Trả về **thời điểm sớm nhất** mà mọi người đều quen biết mọi người khác. Nếu không có thời điểm nào như vậy, trả về `-1`.

**Ví dụ 1:** `logs = [[20190101,0,1],[20190104,3,4],[20190107,2,3],[20190211,1,5],[20190224,2,4],[20190301,0,3],[20190312,1,2],[20190322,4,5]]`, `n = 6` → `20190301`.

**Ví dụ 2:** `logs = [[0,2,0],[1,0,1],[3,0,3],[4,1,2],[7,3,1]]`, `n = 4` → `3`.

**Ràng buộc:** `2 <= n <= 100`; `1 <= logs.length <= 10^4`; `0 <= timestamp_i <= 10^9`; `0 <= x_i, y_i <= n - 1`; `x_i != y_i`; mọi `timestamp` **đôi một khác nhau**; mỗi cặp `(x_i, y_i)` xuất hiện **tối đa một lần**.

In [1]:
import itertools
import heapq
import random
import time
from collections import deque
from typing import List

examples = [
    ([[20190101, 0, 1], [20190104, 3, 4], [20190107, 2, 3], [20190211, 1, 5], [20190224, 2, 4], [20190301, 0, 3], [20190312, 1, 2], [20190322, 4, 5]], 6, 20190301),
    ([[0, 2, 0], [1, 0, 1], [3, 0, 3], [4, 1, 2], [7, 3, 1]], 4, 3),
]
for logs, n, want in examples:
    print(f"n = {n}, {len(logs)} logs -> {want}")

n = 6, 8 logs -> 20190301
n = 4, 5 logs -> 3


## 3. Kiến thức nền

### 3.1 Đồ thị tăng dần

Mỗi log là một **cạnh vô hướng** xuất hiện tại một thời điểm. Tại thời điểm `t`, đồ thị gồm các cạnh có timestamp `≤ t`. "Mọi người quen biết nhau" ⇔ đồ thị **liên thông** (mọi cặp có đường đi). Vì chỉ thêm cạnh nên nếu đồ thị liên thông ở `t` thì cũng liên thông ở mọi `t' ≥ t` (**đơn điệu**).

### 3.2 Union-Find

Cấu trúc `parent[]`: `find(x)` trả về đại diện (gốc) của nhóm chứa `x` (có **nén đường đi**); `union(x, y)` nối hai gốc (có thể theo **kích thước** để cây thấp). `union` trả về `True` nếu **thực sự hợp nhất** hai nhóm khác nhau. Số nhóm ban đầu là `n`, mỗi hợp nhất thực sự giảm `1`.

In [2]:
class DSU:
    def __init__(self, n):
        self.parent = list(range(n))
        self.size = [1] * n
        self.groups = n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]       # nén đường đi (path halving)
            x = self.parent[x]
        return x

    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx == ry:
            return False                                       # đã cùng nhóm
        if self.size[rx] < self.size[ry]:
            rx, ry = ry, rx
        self.parent[ry] = rx
        self.size[rx] += self.size[ry]
        self.groups -= 1
        return True


d = DSU(4)
print(d.union(0, 1), d.union(1, 0), d.union(2, 3), d.groups, d.union(0, 3), d.groups)
assert d.groups == 1

True False True 2 True 1


### 3.3 Liên thông bằng BFS

`connected(n, edges)`: BFS từ người `0`; liên thông ⇔ thăm được cả `n` người.

In [3]:
def connected(n, edges):
    adj = [[] for _ in range(n)]
    for x, y in edges:
        adj[x].append(y)
        adj[y].append(x)
    seen = {0}
    dq = deque([0])
    while dq:
        u = dq.popleft()
        for v in adj[u]:
            if v not in seen:
                seen.add(v)
                dq.append(v)
    return len(seen) == n


assert connected(3, [(0, 1), (1, 2)]) and not connected(3, [(0, 1)])
print("connected ok")

connected ok


## 4. Ý tưởng nền tảng / trực giác

Xem các log theo thứ tự thời gian: ban đầu `n` nhóm rời nhau (mỗi người một nhóm). Mỗi log "nối" hai nhóm (nếu chưa cùng nhóm thì số nhóm giảm `1`). Khi số nhóm còn `1` — mọi người cùng một nhóm — chính là lúc cần tìm, và timestamp của log vừa xử lý là đáp án. Nếu xử lý hết log mà vẫn còn nhiều nhóm thì `-1`.

## 5. Brute force — kiểm tra liên thông sau từng log

**Ý tưởng:** sắp xếp log theo thời gian. Với `i = 0, 1, …`, lấy `i + 1` log đầu và kiểm tra đồ thị có liên thông không bằng BFS; log đầu tiên làm đồ thị liên thông cho đáp án. Đúng theo định nghĩa, làm **oracle**, nhưng `O(m · (n + m))`.

In [4]:
def earliest_brute(logs, n):
    ls = sorted(logs)
    for i in range(len(ls)):
        if connected(n, [(x, y) for _, x, y in ls[:i + 1]]):
            return ls[i][0]
    return -1


for logs, n, want in examples:
    assert earliest_brute(logs, n) == want
print([earliest_brute(l, n) for l, n, _ in examples])

[20190301, 3]


In [5]:
# Code nộp LeetCode cho Brute force (đúng nhưng O(m · (n + m)): chỉ phù hợp khi m nhỏ)
class SolutionBrute:
    def earliestAcq(self, logs: List[List[int]], n: int) -> int:
        logs = sorted(logs)
        for i in range(len(logs)):
            adj = [[] for _ in range(n)]
            for _, x, y in logs[:i + 1]:
                adj[x].append(y)
                adj[y].append(x)
            seen = {0}
            stack = [0]
            while stack:
                u = stack.pop()
                for v in adj[u]:
                    if v not in seen:
                        seen.add(v)
                        stack.append(v)
            if len(seen) == n:
                return logs[i][0]
        return -1

**Độ phức tạp:** `O(m (n + m))` thời gian (với `m = 10^4` tới `10^8`), `O(n + m)` bộ nhớ.

**Nút thắt:** mỗi lần thêm một cạnh, ta dựng lại toàn bộ đồ thị.

## 6. Bước đột phá

Giữ **Union-Find** qua các log đã sắp xếp: mỗi log `O(α(n))`; đếm số nhóm; khi còn `1` thì trả timestamp. `O(m log m)` (sắp xếp) + `O(m α(n))`.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** Union-Find sau khi sắp xếp (đáp án chính), **Cách 3** nhị phân trên chỉ số log + BFS, **Cách 4** Prim (bottleneck của cây khung), **Cách 5** Floyd–Warshall dạng minimax.

### Cách 2 — Union-Find sau khi sắp xếp (đáp án chính)

**Giải thích:** `groups = n`; với từng log (theo thời gian) nếu `union(x, y)` thực sự hợp nhất thì `groups -= 1`; khi `groups == 1` trả timestamp của log đó; sau vòng lặp trả `-1`.

In [6]:
def earliest_dsu(logs, n):
    dsu = DSU(n)
    for t, x, y in sorted(logs):
        dsu.union(x, y)
        if dsu.groups == 1:
            return t
    return -1


for logs, n, want in examples:
    assert earliest_dsu(logs, n) == want
print([earliest_dsu(l, n) for l, n, _ in examples])

[20190301, 3]


In [7]:
# Code nộp LeetCode cho Cách 2
class SolutionDSU:
    def earliestAcq(self, logs: List[List[int]], n: int) -> int:
        parent = list(range(n))

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        groups = n
        for t, x, y in sorted(logs):
            rx, ry = find(x), find(y)
            if rx != ry:
                parent[rx] = ry
                groups -= 1
                if groups == 1:
                    return t
        return -1

**Độ phức tạp:** `O(m log m + m α(n))` thời gian, `O(n)` bộ nhớ.

### Cách 3 — Nhị phân trên chỉ số log + BFS

**Ý tưởng:** theo mục 3.1, "liên thông sau `i + 1` log đầu (đã sắp xếp)" là **đơn điệu** theo `i`. Nhị phân tìm `i` nhỏ nhất thoả; mỗi bước kiểm tra bằng BFS `O(n + m)`.

In [8]:
def earliest_bsearch(logs, n):
    ls = sorted(logs)
    edges = [(x, y) for _, x, y in ls]
    if not connected(n, edges):
        return -1
    lo, hi = 0, len(ls) - 1                               # tìm i nhỏ nhất với connected(edges[:i+1])
    while lo < hi:
        mid = (lo + hi) // 2
        if connected(n, edges[:mid + 1]):
            hi = mid
        else:
            lo = mid + 1
    return ls[lo][0]


for logs, n, want in examples:
    assert earliest_bsearch(logs, n) == want
print([earliest_bsearch(l, n) for l, n, _ in examples])

[20190301, 3]


In [9]:
# Code nộp LeetCode cho Cách 3
class SolutionBSearch:
    def earliestAcq(self, logs: List[List[int]], n: int) -> int:
        logs = sorted(logs)

        def ok(cnt):
            adj = [[] for _ in range(n)]
            for _, x, y in logs[:cnt]:
                adj[x].append(y)
                adj[y].append(x)
            seen = {0}
            stack = [0]
            while stack:
                u = stack.pop()
                for v in adj[u]:
                    if v not in seen:
                        seen.add(v)
                        stack.append(v)
            return len(seen) == n

        if not ok(len(logs)):
            return -1
        lo, hi = 1, len(logs)
        while lo < hi:
            mid = (lo + hi) // 2
            if ok(mid):
                hi = mid
            else:
                lo = mid + 1
        return logs[lo - 1][0]

**Độ phức tạp:** `O(m log m + (n + m) log m)` thời gian, `O(n + m)` bộ nhớ.

### Cách 4 — Prim: cạnh lớn nhất của cây khung nhỏ nhất

**Ý tưởng:** coi timestamp là **trọng số** cạnh. Đáp án (nếu liên thông) bằng **trọng số lớn nhất** trong cây khung nhỏ nhất (MST): đó là ngưỡng `t` nhỏ nhất để các cạnh `≤ t` liên thông (tính chất bottleneck). Chạy Prim từ đỉnh `0` bằng min-heap; nếu không thăm hết `n` đỉnh thì `-1`.

In [10]:
def earliest_prim(logs, n):
    adj = [[] for _ in range(n)]
    for t, x, y in logs:
        adj[x].append((t, y))
        adj[y].append((t, x))
    seen = [False] * n
    heap = [(0, 0)]                                       # (trọng số, đỉnh); đỉnh gốc có trọng số 0
    visited = 0
    best = -1
    while heap:
        w, u = heapq.heappop(heap)
        if seen[u]:
            continue
        seen[u] = True
        visited += 1
        if visited > 1:
            best = max(best, w)                           # cạnh MST vừa thêm
        for t, v in adj[u]:
            if not seen[v]:
                heapq.heappush(heap, (t, v))
    return best if visited == n else -1


for logs, n, want in examples:
    assert earliest_prim(logs, n) == want
print([earliest_prim(l, n) for l, n, _ in examples])

[20190301, 3]


In [11]:
# Code nộp LeetCode cho Cách 4
class SolutionPrim:
    def earliestAcq(self, logs: List[List[int]], n: int) -> int:
        adj = [[] for _ in range(n)]
        for t, x, y in logs:
            adj[x].append((t, y))
            adj[y].append((t, x))
        seen = [False] * n
        heap = [(0, 0)]
        visited, best = 0, -1
        while heap:
            w, u = heapq.heappop(heap)
            if seen[u]:
                continue
            seen[u] = True
            visited += 1
            if visited > 1:
                best = max(best, w)
            for t, v in adj[u]:
                if not seen[v]:
                    heapq.heappush(heap, (t, v))
        return best if visited == n else -1

**Độ phức tạp:** `O(m log m)` thời gian, `O(n + m)` bộ nhớ.

### Cách 5 — Floyd–Warshall dạng minimax

**Ý tưởng:** `d[i][j]` = thời điểm sớm nhất `i` và `j` quen biết nhau = `min` trên các đường đi của `max` timestamp dọc đường. Khởi tạo `d[i][i] = 0`, `d[x][y] = t` cho log, còn lại `∞`; cập nhật `d[i][j] = min(d[i][j], max(d[i][k], d[k][j]))`. Đáp án `= max_{i,j} d[i][j]` (nếu có `∞` thì `-1`). `O(n³)` với `n ≤ 100` (`10^6` bước).

In [12]:
def earliest_floyd(logs, n):
    INF = float("inf")
    d = [[INF] * n for _ in range(n)]
    for i in range(n):
        d[i][i] = -1                                      # một người luôn "quen" chính mình (không ràng buộc thời gian)
    for t, x, y in logs:
        d[x][y] = d[y][x] = min(d[x][y], t)
    for k in range(n):
        dk = d[k]
        for i in range(n):
            dik = d[i][k]
            if dik == INF:
                continue
            di = d[i]
            for j in range(n):
                v = max(dik, dk[j])
                if v < di[j]:
                    di[j] = v
    worst = max(max(row) for row in d)
    return -1 if worst == INF else worst


for logs, n, want in examples:
    assert earliest_floyd(logs, n) == want
print([earliest_floyd(l, n) for l, n, _ in examples])

[20190301, 3]


In [13]:
# Code nộp LeetCode cho Cách 5
class SolutionFloyd:
    def earliestAcq(self, logs: List[List[int]], n: int) -> int:
        INF = float("inf")
        d = [[INF] * n for _ in range(n)]
        for i in range(n):
            d[i][i] = -1
        for t, x, y in logs:
            d[x][y] = d[y][x] = min(d[x][y], t)
        for k in range(n):
            for i in range(n):
                if d[i][k] == INF:
                    continue
                for j in range(n):
                    v = max(d[i][k], d[k][j])
                    if v < d[i][j]:
                        d[i][j] = v
        worst = max(max(row) for row in d)
        return -1 if worst == INF else worst

**Độ phức tạp:** `O(n³ + m)` thời gian (`n ≤ 100`), `O(n²)` bộ nhớ.

## 8. TRACE từng bước

Trace **Cách 2** với ví dụ 2: `n = 4`, `logs` đã sắp xếp theo thời gian: `[0,2,0]`, `[1,0,1]`, `[3,0,3]`, `[4,1,2]`, `[7,3,1]`.

| log | `find(x)`, `find(y)` | hợp nhất? | các nhóm | `groups` |
|---|---|---|---|---|
| (khởi tạo) | | | `{0} {1} {2} {3}` | 4 |
| `t=0: 2-0` | khác nhóm | có | `{0,2} {1} {3}` | 3 |
| `t=1: 0-1` | khác nhóm | có | `{0,1,2} {3}` | 2 |
| `t=3: 0-3` | khác nhóm | có | `{0,1,2,3}` | **1** → trả `3` |

Hàm trace in từng bước:

In [14]:
def trace_dsu(logs, n):
    dsu = DSU(n)
    print(f"n = {n}")
    for t, x, y in sorted(logs):
        merged = dsu.union(x, y)
        comps = {}
        for p in range(n):
            comps.setdefault(dsu.find(p), []).append(p)
        print(f"  t = {t}: {x}-{y} {'hợp nhất' if merged else 'đã cùng nhóm'} -> nhóm {sorted(comps.values())}, groups = {dsu.groups}")
        if dsu.groups == 1:
            print("  => mọi người quen biết nhau tại t =", t)
            return t
    print("  => -1")
    return -1


assert trace_dsu(examples[1][0], 4) == 3
assert trace_dsu(examples[0][0], 6) == 20190301

n = 4
  t = 0: 2-0 hợp nhất -> nhóm [[0, 2], [1], [3]], groups = 3
  t = 1: 0-1 hợp nhất -> nhóm [[0, 1, 2], [3]], groups = 2
  t = 3: 0-3 hợp nhất -> nhóm [[0, 1, 2, 3]], groups = 1
  => mọi người quen biết nhau tại t = 3
n = 6
  t = 20190101: 0-1 hợp nhất -> nhóm [[0, 1], [2], [3], [4], [5]], groups = 5
  t = 20190104: 3-4 hợp nhất -> nhóm [[0, 1], [2], [3, 4], [5]], groups = 4
  t = 20190107: 2-3 hợp nhất -> nhóm [[0, 1], [2, 3, 4], [5]], groups = 3
  t = 20190211: 1-5 hợp nhất -> nhóm [[0, 1, 5], [2, 3, 4]], groups = 2
  t = 20190224: 2-4 đã cùng nhóm -> nhóm [[0, 1, 5], [2, 3, 4]], groups = 2
  t = 20190301: 0-3 hợp nhất -> nhóm [[0, 1, 2, 3, 4, 5]], groups = 1
  => mọi người quen biết nhau tại t = 20190301


**Trace ví dụ 1:** sự kiện `20190224` (`2-4`) không làm giảm số nhóm vì `2` và `4` đã cùng nhóm `{2,3,4}` (xem dòng "đã cùng nhóm"); đến `20190301` (`0-3`) nối `{0,1,5}` với `{2,3,4}` thành một nhóm.

## 9. Code nộp LeetCode

Chọn **Cách 2** (Union-Find sau khi sắp xếp). Giữ nguyên `class Solution`, `earliestAcq(self, logs, n)`:

In [15]:
class Solution:
    def earliestAcq(self, logs: List[List[int]], n: int) -> int:
        parent = list(range(n))

        def find(x: int) -> int:                    # tìm gốc, có nén đường đi
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        groups = n                                  # ban đầu mỗi người một nhóm
        for t, x, y in sorted(logs):                # PHẢI sắp xếp theo thời gian
            rx, ry = find(x), find(y)
            if rx != ry:                            # chỉ khi hai nhóm khác nhau mới hợp nhất
                parent[rx] = ry
                groups -= 1
                if groups == 1:
                    return t
        return -1


print([Solution().earliestAcq(l, n) for l, n, _ in examples])
assert all(Solution().earliestAcq(l, n) == w for l, n, w in examples)

[20190301, 3]


## 10. Vì sao đúng

**(a) Đồ thị theo thời gian.** Gọi `G_t` là đồ thị gồm các log có timestamp `≤ t`. Đáp án là `t*` nhỏ nhất trong số các timestamp sao cho `G_{t*}` liên thông (và `-1` nếu `G_∞` không liên thông). Vì `G_t ⊆ G_{t'}` khi `t ≤ t'`, tính liên thông là đơn điệu. Vì timestamp đôi một khác nhau, mỗi `t` tương ứng đúng một log.

**(b) Union-Find theo thứ tự thời gian.** Sau khi xử lý log `t` (đã sắp xếp), các nhóm của Union-Find chính là các thành phần liên thông của `G_t`. Mỗi hợp nhất thực sự (hai gốc khác nhau) giảm số thành phần đúng `1`; log nối hai người cùng nhóm không đổi số thành phần. `G_t` liên thông ⇔ `groups = 1`. Log đầu tiên làm `groups` về `1` có timestamp `t*`.

**(c) `-1`.** Nếu sau mọi log `groups > 1` thì `G_∞` không liên thông ⇒ `-1`. (Cũng đúng khi `n ≥ 2`; với `n = 1` thì đáp án vô nghĩa, ràng buộc `n ≥ 2`.)

**(d) Nhị phân.** Dùng (a): mệnh đề "`G` sau `i` log đầu liên thông" đơn điệu theo `i` nên nhị phân tìm chỉ số nhỏ nhất; trước đó kiểm tra tổng thể để trả `-1`.

**(e) Prim/bottleneck.** Với mọi ngưỡng `T`: tập đỉnh liên thông bởi các cạnh `≤ T` là hợp các thành phần của `G_T`. Trong MST bất kỳ, đường đi giữa hai đỉnh là đường **minimax** (cực tiểu của cạnh lớn nhất). Do đó cạnh lớn nhất của MST = `min` ngưỡng `T` để toàn bộ liên thông = `t*`. Prim sinh một MST của thành phần chứa đỉnh `0`; nếu thành phần này không phủ hết `n` đỉnh thì không liên thông ⇒ `-1`.

**(f) Floyd–Warshall minimax.** `d[i][j]` là giá trị `min` của `max` timestamp trên đường đi: cập nhật `d[i][j] = min(d[i][j], max(d[i][k], d[k][j]))` đúng cho phép toán `(min, max)` (nửa vòng). `max_{i,j} d[i][j]` là ngưỡng nhỏ nhất để mọi cặp thông nhau.

**(g) Kiểm chứng bằng máy.** Mọi cách khớp vét cạn (kiểm tra liên thông sau từng log) trên mọi dãy cạnh có thứ tự của đồ thị `≤ 4` đỉnh (kể cả thứ tự đầu vào bị xáo) và trên các đồ thị ngẫu nhiên.

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Kiểm tra từng tiền tố | `O(m (n + m))` | `O(n + m)` | oracle, `m` nhỏ |
| 2. Union-Find | `O(m log m + m α(n))` | `O(n)` | **đáp án chính** |
| 3. Nhị phân + BFS | `O(m log m + (n + m) log m)` | `O(n + m)` | dùng tính đơn điệu |
| 4. Prim | `O(m log m)` | `O(n + m)` | bottleneck của MST |
| 5. Floyd–Warshall | `O(n³ + m)` | `O(n²)` | `n ≤ 100` |

Đo thực tế (`n = 100`, `m = 4 950` cạnh, mọi cặp):

In [16]:
def complete_logs(rng, n):
    pairs = [(x, y) for x in range(n) for y in range(x + 1, n)]
    ts = rng.sample(range(10 ** 9 + 1), len(pairs))
    logs = [[t, x, y] for t, (x, y) in zip(ts, pairs)]
    rng.shuffle(logs)
    return logs


def bench(fn, logs, n, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(logs, n)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(1101)
big = complete_logs(rng, 100)
for name, fn in [("dsu", earliest_dsu), ("bsearch", earliest_bsearch), ("prim", earliest_prim), ("floyd", earliest_floyd)]:
    print(f"{name:8s}: {bench(fn, big, 100) * 1000:8.2f} ms -> {fn(big, 100)}")

dsu     :     1.93 ms -> 59007414
bsearch :     3.21 ms -> 59007414
prim    :     3.24 ms -> 59007414


floyd   :    60.51 ms -> 59007414


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi dãy cạnh nhỏ, (E) tính chất / ngẫu nhiên (seed cố định), (F) giới hạn.

### A. Ví dụ của đề

In [17]:
IMPLS = {
    "dsu": earliest_dsu,
    "bsearch": earliest_bsearch,
    "prim": earliest_prim,
    "floyd": earliest_floyd,
    "Solution": lambda l, n: Solution().earliestAcq([r[:] for r in l], n),
    "SolutionDSU": lambda l, n: SolutionDSU().earliestAcq([r[:] for r in l], n),
    "SolutionBSearch": lambda l, n: SolutionBSearch().earliestAcq([r[:] for r in l], n),
    "SolutionPrim": lambda l, n: SolutionPrim().earliestAcq([r[:] for r in l], n),
    "SolutionFloyd": lambda l, n: SolutionFloyd().earliestAcq([r[:] for r in l], n),
}
BRUTE = {"brute": earliest_brute, "SolutionBrute": lambda l, n: SolutionBrute().earliestAcq([r[:] for r in l], n)}


def check(logs, n, want=None, brute=True):
    if want is None:
        want = earliest_brute(logs, n) if brute else Solution().earliestAcq([r[:] for r in logs], n)
    impls = dict(IMPLS)
    if brute:
        impls.update(BRUTE)
    for name, fn in impls.items():
        got = fn(logs, n)
        assert got == want, (name, n, logs, got, want)
    return want


for logs, n, want in examples:
    check(logs, n, want, brute=len(logs) <= 20)
print("ví dụ đề: mọi cài đặt đúng;", len(IMPLS), "cách nhanh +", len(BRUTE), "vét cạn")

ví dụ đề: mọi cài đặt đúng; 9 cách nhanh + 2 vét cạn


### B. Edge case

- `n = 2`, một log: đáp án là timestamp của log đó.
- Không bao giờ liên thông: `-1` (có một người cô lập, hoặc hai nhóm tách rời).
- Log **không sắp xếp** theo thời gian (đảo ngược).
- Log cuối là log duy nhất làm liên thông.
- Có log "thừa" (cùng nhóm) sau khi đã liên thông: đáp án vẫn là log hoàn tất sớm nhất.
- Timestamp `0` và `10^9`.
- Đồ thị đường thẳng cần mọi cạnh; đồ thị sao.
- Hai thành phần lớn, chỉ một cạnh nối chúng.

In [18]:
edge = [
    ([[5, 0, 1]], 2, 5),
    ([[5, 0, 1]], 3, -1),
    ([[3, 0, 1], [2, 1, 2]], 3, 3),
    ([[2, 1, 2], [3, 0, 1]], 3, 3),                         # đảo thứ tự đầu vào
    ([[1, 0, 1], [2, 2, 3]], 4, -1),                        # hai nhóm tách rời
    ([[0, 0, 1], [10 ** 9, 1, 2]], 3, 10 ** 9),
    ([[1, 0, 1], [2, 1, 2], [3, 0, 2], [4, 2, 3]], 4, 4),
    ([[1, 0, 1], [2, 0, 2], [3, 0, 3], [4, 0, 4], [5, 1, 2]], 5, 4),
    ([[1, 0, 1], [2, 2, 3], [3, 1, 2]], 4, 3),
    ([[7, 0, 1], [1, 2, 3], [4, 0, 2], [3, 1, 3]], 4, 4),
]
for logs, n, want in edge:
    got = check(logs, n, want)
    print(f"  n = {n}, {len(logs)} logs -> {got}")
print(len(edge), "edge case: mọi cài đặt (kể cả vét cạn) khớp")

  n = 2, 1 logs -> 5
  n = 3, 1 logs -> -1
  n = 3, 2 logs -> 3
  n = 3, 2 logs -> 3
  n = 4, 2 logs -> -1
  n = 3, 2 logs -> 1000000000
  n = 4, 4 logs -> 4
  n = 5, 5 logs -> 4
  n = 4, 3 logs -> 3
  n = 4, 4 logs -> 4
10 edge case: mọi cài đặt (kể cả vét cạn) khớp


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [19]:
def bug_no_sort(logs, n):                               # không sắp xếp log theo thời gian
    dsu = DSU(n)
    for t, x, y in logs:
        dsu.union(x, y)
        if dsu.groups == 1:
            return t
    return -1


def bug_always_decrement(logs, n):                      # giảm số nhóm cả khi hai người đã cùng nhóm
    dsu = DSU(n)
    groups = n
    for t, x, y in sorted(logs):
        dsu.union(x, y)
        groups -= 1
        if groups == 1:
            return t
    return -1


def bug_no_minus_one(logs, n):                          # không phát hiện trường hợp không bao giờ liên thông
    dsu = DSU(n)
    last = -1
    for t, x, y in sorted(logs):
        dsu.union(x, y)
        last = t
        if dsu.groups == 1:
            return t
    return last


def bug_no_find(logs, n):                               # gộp trực tiếp parent[x] = y, không tìm gốc
    parent = list(range(n))
    for t, x, y in sorted(logs):
        parent[x] = y
        if len({parent[i] if parent[i] == i else -1 for i in range(n)} - {-1}) == 1:
            return t
    return -1


def bug_return_index(logs, n):                          # trả về chỉ số log thay vì timestamp
    dsu = DSU(n)
    for i, (t, x, y) in enumerate(sorted(logs)):
        dsu.union(x, y)
        if dsu.groups == 1:
            return i
    return -1


def bug_edges_count(logs, n):                           # nghĩ rằng cần đúng n - 1 log đầu tiên
    ls = sorted(logs)
    return ls[n - 2][0] if len(ls) >= n - 1 else -1


BUGS = [bug_no_sort, bug_always_decrement, bug_no_minus_one, bug_no_find, bug_return_index, bug_edges_count]


def random_logs(rng, n, m, max_t=100):
    pairs = [(x, y) for x in range(n) for y in range(x + 1, n)]
    rng.shuffle(pairs)
    pairs = [(y, x) if rng.random() < 0.5 else (x, y) for x, y in pairs[:min(m, len(pairs))]]
    ts = rng.sample(range(max_t + 1), len(pairs))
    return [[t, x, y] for t, (x, y) in zip(ts, pairs)]


rng = random.Random(11010)
pool = []
for _ in range(3000):
    n = rng.randint(2, 5)
    pool.append((random_logs(rng, n, rng.randint(1, n * (n - 1) // 2)), n))
for bug in BUGS:
    found = next(((l, n) for l, n in pool if bug(l, n) != earliest_brute(l, n)), None)
    assert found, bug.__name__
    l, n = found
    print(f"{bug.__name__:20s} sai trên n = {n}, logs = {l}: ra {bug(l, n)}, đúng {earliest_brute(l, n)}")

bug_no_sort          sai trên n = 5, logs = [[60, 2, 1], [53, 3, 2], [32, 1, 0], [46, 3, 1], [26, 4, 0]]: ra 26, đúng 53
bug_always_decrement sai trên n = 4, logs = [[94, 2, 3], [14, 1, 3], [17, 2, 1]]: ra 94, đúng -1
bug_no_minus_one     sai trên n = 3, logs = [[52, 1, 2]]: ra 52, đúng -1
bug_no_find          sai trên n = 5, logs = [[60, 2, 1], [53, 3, 2], [32, 1, 0], [46, 3, 1], [26, 4, 0]]: ra 60, đúng 53
bug_return_index     sai trên n = 5, logs = [[60, 2, 1], [53, 3, 2], [32, 1, 0], [46, 3, 1], [26, 4, 0]]: ra 3, đúng 53
bug_edges_count      sai trên n = 4, logs = [[94, 2, 3], [14, 1, 3], [17, 2, 1]]: ra 94, đúng -1


### D. Vét cạn mọi dãy cạnh nhỏ

Mọi dãy cạnh **có thứ tự** (ứng với timestamp tăng dần) trên `n = 2, 3, 4` đỉnh, độ dài `1..C(n,2)` (với `n = 4`: `1 957` dãy). Mỗi dãy kiểm tra ở hai dạng: sắp xếp sẵn và **đảo ngược/xáo trộn** thứ tự đầu vào. So mọi cài đặt với vét cạn.

In [20]:
t0 = time.perf_counter()
cnt = 0
rs = random.Random(3)
for n in (2, 3, 4):
    pairs = [(x, y) for x in range(n) for y in range(x + 1, n)]
    for k in range(1, len(pairs) + 1):
        for seq in itertools.permutations(pairs, k):
            logs = [[10 * (i + 1) + 3, x, y] for i, (x, y) in enumerate(seq)]
            want = earliest_brute(logs, n)
            shuffled = logs[:]
            rs.shuffle(shuffled)
            for name, fn in IMPLS.items():
                assert fn(logs, n) == want and fn(shuffled, n) == want, (name, n, seq)
            cnt += 1
print(cnt, "dãy cạnh có thứ tự:", len(IMPLS), "cài đặt khớp vét cạn (đã sắp xếp và đã xáo);", round(time.perf_counter() - t0, 1), "s")

1972 dãy cạnh có thứ tự: 9 cài đặt khớp vét cạn (đã sắp xếp và đã xáo); 0.2 s


### E. Tính chất / ngẫu nhiên (seed cố định)

(1) Đồ thị ngẫu nhiên nhỏ (`n ≤ 8`): mọi cách khớp vét cạn; (2) `n` lớn: các cách khớp nhau; (3) đáp án là một **timestamp có mặt** hoặc `-1`; `-1` ⇔ đồ thị mọi log **không liên thông**; (4) **hoán vị log** không đổi; (5) **đổi nhãn người** không đổi; (6) **biến đổi tăng ngặt timestamp** `f` ánh xạ đáp án: `ans(f(t)) = f(ans(t))`; (7) **thêm một log** không làm đáp án tăng (nếu đã liên thông) và không biến liên thông thành `-1`; (8) đáp án `≥` thời điểm xuất hiện log thứ `n - 1` theo thời gian (cần ít nhất `n - 1` cạnh).

In [21]:
rng = random.Random(20240201)
t0 = time.perf_counter()
for it in range(500):
    n = rng.randint(2, 8)
    logs = random_logs(rng, n, rng.randint(1, n * (n - 1) // 2), max_t=rng.choice([50, 1000]))
    check(logs, n)                                                                    # (1)
print("(1) 500 đồ thị n <= 8 khớp vét cạn:", round(time.perf_counter() - t0, 1), "s")

t0 = time.perf_counter()
for it in range(200):
    n = rng.randint(2, 40)
    m = rng.randint(1, min(n * (n - 1) // 2, 300))
    logs = random_logs(rng, n, m, max_t=10 ** 9)
    ans = check(logs, n, brute=False)                                                 # (2)
    times = {t for t, _, _ in logs}
    assert ans == -1 or ans in times                                                  # (3)
    assert (ans == -1) == (not connected(n, [(x, y) for _, x, y in logs]))
    sh = [r[:] for r in logs]
    rng.shuffle(sh)
    assert Solution().earliestAcq(sh, n) == ans                                       # (4)
    perm = list(range(n))
    rng.shuffle(perm)
    assert Solution().earliestAcq([[t, perm[x], perm[y]] for t, x, y in logs], n) == ans   # (5)
    a, b = rng.randint(2, 7), rng.randint(0, 10)
    assert Solution().earliestAcq([[a * t + b, x, y] for t, x, y in logs], n) == (-1 if ans == -1 else a * ans + b)   # (6)
    all_pairs = {(min(x, y), max(x, y)) for _, x, y in logs}
    cand = [(x, y) for x in range(n) for y in range(x + 1, n) if (x, y) not in all_pairs]
    if cand:
        x, y = rng.choice(cand)
        t_new = max(times) + rng.randint(1, 10)
        ans2 = Solution().earliestAcq(logs + [[t_new, x, y]], n)
        assert (ans != -1 and ans2 == ans) or (ans == -1 and ans2 in (-1, t_new))     # (7)
    if ans != -1:
        assert ans >= sorted(times)[n - 2]                                            # (8)
print("(2)-(8) 200 đồ thị n <= 40:", round(time.perf_counter() - t0, 1), "s")

(1) 500 đồ thị n <= 8 khớp vét cạn: 0.1 s


(2)-(8) 200 đồ thị n <= 40: 0.7 s


### F. Giới hạn

`n = 100`: đồ thị đầy đủ `4 950` cạnh (timestamp ngẫu nhiên tới `10^9`); đường thẳng (đáp án là cạnh lớn nhất); sao; có một người cô lập (`-1`); hai nửa nối bằng một cạnh duy nhất; logs đảo ngược thứ tự. Mọi cách nhanh khớp nhau.

In [22]:
t0 = time.perf_counter()
rg = random.Random(77)
N = 100
cases = {}
cases["đầy đủ"] = (complete_logs(rg, N), N)
ts = rg.sample(range(10 ** 9 + 1), N - 1)
cases["đường thẳng"] = ([[t, i, i + 1] for i, t in enumerate(ts)], N)
ts = rg.sample(range(10 ** 9 + 1), N - 1)
cases["sao"] = ([[t, 0, i + 1] for i, t in enumerate(ts)], N)
ts = rg.sample(range(10 ** 9 + 1), N - 2)
cases["một người cô lập"] = ([[t, i, i + 1] for i, t in enumerate(ts)], N)
half = []
ts = iter(rg.sample(range(10 ** 9 + 1), 200))
for base in (0, 50):
    for i in range(49):
        half.append([next(ts), base + i, base + i + 1])
half.append([next(ts), 49, 50])
cases["hai nửa + cầu"] = (half, N)
cases["đảo ngược"] = (list(reversed(sorted(cases["đường thẳng"][0]))), N)
for name, (logs, n) in cases.items():
    want = Solution().earliestAcq([r[:] for r in logs], n)
    for iname, fn in IMPLS.items():
        assert fn(logs, n) == want, (iname, name)
    print(f"  {name:18s}: {len(logs):5d} logs -> {want}")
assert Solution().earliestAcq(cases["một người cô lập"][0], N) == -1
line = cases["đường thẳng"][0]
assert Solution().earliestAcq(line, N) == max(t for t, _, _ in line)
half_ans = Solution().earliestAcq(half, N)
assert half_ans == max(t for t, _, _ in half)
print("n = 100:", round(time.perf_counter() - t0, 1), "s")

  đầy đủ            :  4950 logs -> 37415928


  đường thẳng       :    99 logs -> 998180974
  sao               :    99 logs -> 984713742


  một người cô lập  :    98 logs -> -1


  hai nửa + cầu     :    99 logs -> 982976821
  đảo ngược         :    99 logs -> 998180974
n = 100: 0.6 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: không sắp xếp log theo thời gian

In [23]:
l, n = [[9, 0, 1], [1, 1, 2]], 3
print("bug:", bug_no_sort(l, n), "| đúng:", Solution().earliestAcq(l, n))
assert bug_no_sort(l, n) == 1 and Solution().earliestAcq(l, n) == 9
bad = next((l, n) for l, n in pool if bug_no_sort(l, n) != earliest_brute(l, n))
print("phản ví dụ:", bad, "| bug:", bug_no_sort(*bad), "| đúng:", earliest_brute(*bad))
assert bug_no_sort(*bad) != earliest_brute(*bad)

bug: 1 | đúng: 9
phản ví dụ: ([[60, 2, 1], [53, 3, 2], [32, 1, 0], [46, 3, 1], [26, 4, 0]], 5) | bug: 26 | đúng: 53


**Vì sao sai:** đề không đảm bảo `logs` đã theo thứ tự thời gian. Xử lý theo thứ tự đầu vào cho thời điểm sai. Luôn `sorted(logs)` trước.

### Lỗi 2: giảm số nhóm cả khi hai người đã cùng nhóm

In [24]:
bad = next((l, n) for l, n in pool if bug_always_decrement(l, n) != earliest_brute(l, n))
print("phản ví dụ:", bad, "| bug:", bug_always_decrement(*bad), "| đúng:", earliest_brute(*bad))
assert bug_always_decrement(*bad) != earliest_brute(*bad)

phản ví dụ: ([[94, 2, 3], [14, 1, 3], [17, 2, 1]], 4) | bug: 94 | đúng: -1


**Vì sao sai:** log nối hai người đã cùng nhóm (đã quen biết bắc cầu) **không** làm số nhóm giảm. Chỉ giảm khi `find(x) != find(y)`.

### Lỗi 3: quên trả `-1`

In [25]:
l, n = [[1, 0, 1]], 3
print("bug:", bug_no_minus_one(l, n), "| đúng:", Solution().earliestAcq(l, n))
assert bug_no_minus_one(l, n) == 1 and Solution().earliestAcq(l, n) == -1

bug: 1 | đúng: -1


**Vì sao sai:** nếu sau mọi log vẫn còn nhiều hơn một nhóm thì không có thời điểm nào thoả: trả `-1`.

### Lỗi 4: gộp mà không tìm gốc

In [26]:
bad = next((l, n) for l, n in pool if bug_no_find(l, n) != earliest_brute(l, n))
print("phản ví dụ:", bad, "| bug:", bug_no_find(*bad), "| đúng:", earliest_brute(*bad))
assert bug_no_find(*bad) != earliest_brute(*bad)

phản ví dụ: ([[60, 2, 1], [53, 3, 2], [32, 1, 0], [46, 3, 1], [26, 4, 0]], 5) | bug: 60 | đúng: 53


**Vì sao sai:** `parent[x] = y` chỉ nối **người** `x`, không nối **cả nhóm** của `x`. Phải nối **gốc** của hai nhóm: `parent[find(x)] = find(y)`.

### Lỗi 5: trả về chỉ số thay vì timestamp

In [27]:
l, n = examples[1][0], examples[1][1]
print("bug:", bug_return_index(l, n), "| đúng:", Solution().earliestAcq(l, n))
assert bug_return_index(l, n) == 2 and Solution().earliestAcq(l, n) == 3

bug: 2 | đúng: 3


**Vì sao sai:** đề hỏi **thời điểm** (timestamp), không phải thứ tự log.

### Lỗi 6 (bonus): cho rằng cần đúng `n - 1` log đầu tiên

In [28]:
bad = next((l, n) for l, n in pool if bug_edges_count(l, n) != earliest_brute(l, n))
print("phản ví dụ:", bad, "| bug:", bug_edges_count(*bad), "| đúng:", earliest_brute(*bad))
assert bug_edges_count(*bad) != earliest_brute(*bad)

phản ví dụ: ([[94, 2, 3], [14, 1, 3], [17, 2, 1]], 4) | bug: 94 | đúng: -1


**Vì sao sai:** có thể có log "thừa" (nối hai người đã cùng nhóm) nên cần **nhiều hơn `n - 1`** log; và có thể không đủ để liên thông dù có `≥ n - 1` log.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "các sự kiện thêm cạnh theo thời gian, hỏi thời điểm đầu tiên đồ thị liên thông / số thành phần bằng `1`" → **sắp xếp + Union-Find** (hoặc Kruskal/bottleneck MST).

**Mẹo:**

1. Sắp xếp log theo timestamp trước.
2. Chỉ giảm số nhóm khi `find(x) != find(y)`.
3. Dừng sớm khi `groups == 1`; hết log mà còn nhiều nhóm ⇒ `-1`.
4. Đáp án = cạnh lớn nhất của MST (bottleneck).
5. Tính đơn điệu cho phép nhị phân theo thời gian.
6. Oracle: BFS sau mỗi log trên đồ thị nhỏ.

**Họ bài cùng pattern:** Number of Provinces (547), Redundant Connection (684), Connecting Cities With Minimum Cost (1135), Path With Minimum Effort (1631), Swim in Rising Water (778), Number of Operations to Make Network Connected (1319).

## 15. Biến thể

1. **Số nhóm tại thời điểm `T`.**
2. **Thời điểm sớm nhất hai người cụ thể `a`, `b` quen biết nhau** (minimax giữa `a` và `b`).
3. **Kích thước nhóm lớn nhất theo thời gian** (dãy các giá trị sau mỗi log).
4. **Số log "thừa"** (không làm giảm số nhóm) tới thời điểm liên thông.

Cài đặt và đối chiếu BFS:

In [29]:
def groups_at(logs, n, T):
    dsu = DSU(n)
    for t, x, y in sorted(logs):
        if t <= T:
            dsu.union(x, y)
    return dsu.groups


def earliest_pair(logs, n, a, b):
    dsu = DSU(n)
    if a == b:
        return -1
    for t, x, y in sorted(logs):
        dsu.union(x, y)
        if dsu.find(a) == dsu.find(b):
            return t
    return -1


def largest_group_over_time(logs, n):
    dsu = DSU(n)
    out = []
    for t, x, y in sorted(logs):
        dsu.union(x, y)
        out.append((t, max(dsu.size[dsu.find(i)] for i in range(n))))
    return out


def redundant_logs(logs, n):
    dsu = DSU(n)
    cnt = 0
    for t, x, y in sorted(logs):
        if not dsu.union(x, y):
            cnt += 1
        if dsu.groups == 1:
            return cnt
    return None


def bfs_components(n, edges):
    adj = [[] for _ in range(n)]
    for x, y in edges:
        adj[x].append(y)
        adj[y].append(x)
    seen, comps = set(), []
    for s in range(n):
        if s in seen:
            continue
        seen.add(s)
        comp, dq = [s], deque([s])
        while dq:
            u = dq.popleft()
            for v in adj[u]:
                if v not in seen:
                    seen.add(v)
                    comp.append(v)
                    dq.append(v)
        comps.append(comp)
    return comps


rng = random.Random(15)
for _ in range(400):
    n = rng.randint(2, 8)
    logs = random_logs(rng, n, rng.randint(1, n * (n - 1) // 2), max_t=60)
    for T in (0, 20, 40, 60):
        edges = [(x, y) for t, x, y in logs if t <= T]
        assert groups_at(logs, n, T) == len(bfs_components(n, edges))
    a, b = rng.sample(range(n), 2)
    want = -1
    for i in range(len(logs)):
        edges = [(x, y) for _, x, y in sorted(logs)[:i + 1]]
        comp = next(c for c in bfs_components(n, edges) if a in c)
        if b in comp:
            want = sorted(logs)[i][0]
            break
    assert earliest_pair(logs, n, a, b) == want
    for t, size in largest_group_over_time(logs, n):
        edges = [(x, y) for tt, x, y in logs if tt <= t]
        assert size == max(len(c) for c in bfs_components(n, edges))
    ans = earliest_brute(logs, n)
    if ans != -1:
        used = [x for x in sorted(logs) if x[0] <= ans]
        assert redundant_logs(logs, n) == len(used) - (n - 1)
print("số nhóm tại T / thời điểm hai người quen nhau / nhóm lớn nhất / log thừa: khớp BFS")

số nhóm tại T / thời điểm hai người quen nhau / nhóm lớn nhất / log thừa: khớp BFS


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS từng tiền tố | định nghĩa | `O(m (n + m))` | `O(n + m)` | oracle |
| 2. Union-Find | sắp xếp + hợp nhất | `O(m log m)` | `O(n)` | **đáp án chính** |
| 3. Nhị phân + BFS | đơn điệu theo thời gian | `O(m log m)` | `O(n + m)` | khi chỉ có "kiểm tra liên thông" |
| 4. Prim | bottleneck MST | `O(m log m)` | `O(n + m)` | cách nhìn cây khung |
| 5. Floyd–Warshall | minimax | `O(n³)` | `O(n²)` | `n ≤ 100`, nhiều truy vấn cặp |

Mọi cách khớp vét cạn trên `2 000+` dãy cạnh có thứ tự (hai thứ tự đầu vào) và `500` đồ thị ngẫu nhiên; khớp nhau trên `n = 100`.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [30]:
print(Solution().earliestAcq([[1, 0, 1], [2, 1, 2]], 3), Solution().earliestAcq([[2, 0, 1], [1, 1, 2], [3, 0, 2]], 3), Solution().earliestAcq([[1, 0, 1], [2, 2, 3]], 4), Solution().earliestAcq([[5, 0, 1]], 2))

2 2 -1 5


### Bài 2 (tính tay)

`n = 5`, `logs = [[4,0,1],[1,2,3],[7,1,2],[3,3,4],[9,0,4],[2,1,4]]` (chưa sắp xếp). Sắp xếp theo thời gian, lập bảng nhóm sau mỗi log và cho biết đáp án. Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện Union-Find:

In [31]:
def earliest_todo(logs, n):
    parent = list(range(n))

    def find(x):
        return x        # TODO 1: tìm gốc (có nén đường đi)

    groups = n
    for t, x, y in sorted(logs):
        pass            # TODO 2: nếu find(x) != find(y): hợp nhất, groups -= 1; nếu groups == 1 trả t
    return -1


# Khi làm xong, bỏ comment để kiểm tra:
# assert all(earliest_todo(l, n) == w for l, n, w in examples)

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **Prim** (bottleneck của MST):

In [32]:
class MySolution:
    def earliestAcq(self, logs, n):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().earliestAcq(l, n) == w for l, n, w in examples)

### Bài 5 (mini challenge): thời điểm hai người cụ thể quen nhau

Viết `earliest_between(logs, n, a, b)` trả về timestamp sớm nhất mà `a` và `b` quen biết nhau (bắc cầu), hoặc `-1`. Ví dụ với ví dụ 2: `earliest_between(logs, 4, 1, 2) == 1` (`2-0` lúc `0`, `0-1` lúc `1`).

In [33]:
def earliest_between(logs, n, a, b):
    pass            # TODO


# Khi làm xong:
# logs2 = examples[1][0]
# assert earliest_between(logs2, 4, 1, 2) == 1 and earliest_between(logs2, 4, 0, 3) == 3

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: sắp xếp log theo thời gian rồi gộp nhóm.
- Mức 2: `[[2,0,1],[1,1,2],[3,0,2]]`: sau `t=1` có `{1,2}`; sau `t=2` có `{0,1,2}`; `[[1,0,1],[2,2,3]]` với `n = 4`: hai nhóm tách rời.
- Mức 3: `2 2 -1 5`.

**Bài 2**

- Mức 1: sắp xếp: `1: 2-3`, `2: 1-4`, `3: 3-4`, `4: 0-1`, `7: 1-2`, `9: 0-4`.
- Mức 2: nhóm: `{2,3}`; `{2,3},{1,4}`; `{1,2,3,4}`; `{0,1,2,3,4}` tại `t = 4`.
- Mức 3: đáp án `4`.

**Bài 3**

- Mức 1: `while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]; return x`.
- Mức 2: `rx, ry = find(x), find(y)`; nếu khác nhau: `parent[rx] = ry`; `groups -= 1`.
- Mức 3: `if groups == 1: return t`.

**Bài 4**

- Mức 1: dựng danh sách kề với `(t, v)`.
- Mức 2: Prim từ `0` bằng min-heap, ghi `best = max(best, w)` mỗi lần thăm đỉnh mới.
- Mức 3: nếu số đỉnh thăm `< n` trả `-1`.

**Bài 5**

- Mức 1: Union-Find theo thứ tự thời gian.
- Mức 2: sau mỗi hợp nhất, kiểm tra `find(a) == find(b)`.
- Mức 3: trả timestamp đầu tiên thoả; không có thì `-1`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [34]:
vals = (Solution().earliestAcq([[1, 0, 1], [2, 1, 2]], 3), Solution().earliestAcq([[2, 0, 1], [1, 1, 2], [3, 0, 2]], 3), Solution().earliestAcq([[1, 0, 1], [2, 2, 3]], 4), Solution().earliestAcq([[5, 0, 1]], 2))
print(vals)
assert vals == (2, 2, -1, 5)
assert vals == (earliest_brute([[1, 0, 1], [2, 1, 2]], 3), earliest_brute([[2, 0, 1], [1, 1, 2], [3, 0, 2]], 3), earliest_brute([[1, 0, 1], [2, 2, 3]], 4), earliest_brute([[5, 0, 1]], 2))
print("Bài 1 OK")

(2, 2, -1, 5)
Bài 1 OK


### Bài 2

In [35]:
l5 = [[4, 0, 1], [1, 2, 3], [7, 1, 2], [3, 3, 4], [9, 0, 4], [2, 1, 4]]
assert trace_dsu(l5, 5) == 4 == earliest_brute(l5, 5)
print("Bài 2 OK")

n = 5
  t = 1: 2-3 hợp nhất -> nhóm [[0], [1], [2, 3], [4]], groups = 4
  t = 2: 1-4 hợp nhất -> nhóm [[0], [1, 4], [2, 3]], groups = 3
  t = 3: 3-4 hợp nhất -> nhóm [[0], [1, 2, 3, 4]], groups = 2
  t = 4: 0-1 hợp nhất -> nhóm [[0, 1, 2, 3, 4]], groups = 1
  => mọi người quen biết nhau tại t = 4
Bài 2 OK


### Bài 3

In [36]:
def earliest_answer(logs, n):
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    groups = n
    for t, x, y in sorted(logs):
        rx, ry = find(x), find(y)
        if rx != ry:
            parent[rx] = ry
            groups -= 1
            if groups == 1:
                return t
    return -1


assert all(earliest_answer(l, n) == w for l, n, w in examples)
for l, n in pool:
    assert earliest_answer(l, n) == earliest_brute(l, n)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [37]:
class MySolutionAnswer:
    def earliestAcq(self, logs, n):
        adj = [[] for _ in range(n)]
        for t, x, y in logs:
            adj[x].append((t, y))
            adj[y].append((t, x))
        seen, heap, count, best = [False] * n, [(0, 0)], 0, -1
        while heap:
            w, u = heapq.heappop(heap)
            if seen[u]:
                continue
            seen[u] = True
            count += 1
            if count > 1:
                best = max(best, w)
            for t, v in adj[u]:
                if not seen[v]:
                    heapq.heappush(heap, (t, v))
        return best if count == n else -1


assert all(MySolutionAnswer().earliestAcq(l, n) == w for l, n, w in examples)
for l, n in pool:
    assert MySolutionAnswer().earliestAcq(l, n) == earliest_brute(l, n)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [38]:
def earliest_between(logs, n, a, b):
    return earliest_pair(logs, n, a, b)


logs2 = examples[1][0]
assert earliest_between(logs2, 4, 1, 2) == 1 and earliest_between(logs2, 4, 0, 3) == 3
for l, n in pool[:500]:
    a, b = 0, n - 1
    want = -1
    for i in range(len(l)):
        edges = [(x, y) for _, x, y in sorted(l)[:i + 1]]
        if any(a in c and b in c for c in bfs_components(n, edges)):
            want = sorted(l)[i][0]
            break
    assert earliest_between(l, n, a, b) == want
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (thời điểm đồ thị liên thông khi các cạnh xuất hiện theo thời gian):**

```python
class Solution:
    def earliestAcq(self, logs: List[List[int]], n: int) -> int:
        parent = list(range(n))

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        groups = n
        for t, x, y in sorted(logs):
            rx, ry = find(x), find(y)
            if rx != ry:
                parent[rx] = ry
                groups -= 1
                if groups == 1:
                    return t
        return -1
```

**Checklist trước khi submit:**

- [ ] Đã `sorted(logs)` theo timestamp (đầu vào không đảm bảo thứ tự)?
- [ ] Chỉ giảm `groups` khi `find(x) != find(y)`; nối **gốc** với gốc?
- [ ] Trả timestamp (không phải chỉ số) khi `groups == 1`?
- [ ] Hết log mà `groups > 1` ⇒ `-1`?
- [ ] Đã thử log thừa, log đảo thứ tự, người cô lập, `n = 2`?
- [ ] Đúng tên `class Solution`, `earliestAcq(self, logs, n)`, không in debug.